#### ENRICH `ORDER_ITEMS` FROM BRONZE TO SILVER

In [0]:
order_items_df = spark.table('ORDERS_SALES.BRONZE.ORDER_ITEMS')
order_items_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

# Select and cast columns properly
order_items_enriched_df = (
    order_items_df
    .select(
        trim(col("order_id")).alias("order_id"),
        col("order_item_id").cast("int").alias("order_item_id"),
        trim(col("product_id")).alias("product_id"),
        trim(col("seller_id")).alias("seller_id"),
        to_timestamp(col("shipping_limit_date")).alias("shipping_limit_date"),
        col("price").cast("double").alias("price"),
        col("freight_value").cast("double").alias("freight_value"),
        col("file_path")
        
)
)
# Show one row for verification
order_items_enriched_df.limit(1).display()

In [0]:
from pyspark.sql.window import Window

w = (
    Window.partitionBy("order_id", "order_item_id")
    .orderBy(col("shipping_limit_date").desc_nulls_last(), col("price").desc_nulls_last())
)

nodup_df = order_items_enriched_df.withColumn("rn", row_number().over(w)).where(col("rn") == 1).drop(col("rn"))

nodup_df.limit(1).display()

In [0]:
flagged_df = (nodup_df.select("order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value", "file_path")
                      .withColumn("is_price_non_positive", col("price") <= 0)
                      .withColumn("is_freight_negative", col("freight_value") <= 0)
                      .withColumn("is_ship_date_outlier", (year(col("shipping_limit_date")) < lit(2017)) | (year(col("shipping_limit_date")) > lit(2019)))
                      )

flagged_df.limit(1).display()

In [0]:
order_items_trans_df = (
    flagged_df
    .withColumn("total_item_value", round(col("price") + col("freight_value")))
    .withColumn(
        "freight_ratio", 
        when(col("price") > 0, round(col("freight_value") / col("price"), 3)).otherwise(lit(None))
    )
    .withColumn("ship_year", year(col("shipping_limit_date")))
    .withColumn("ship_month", month(col("shipping_limit_date")))
    .withColumn("ship_date", dayofmonth(col("shipping_limit_date")))
)
order_items_trans_df.limit(1).display()

In [0]:
order_items_enriched_df = (order_items_trans_df
                           .select("order_id", "order_item_id", "product_id", "seller_id", 
                                   "shipping_limit_date", "price", "freight_value", "is_price_non_positive", "is_freight_negative", "is_ship_date_outlier", "total_item_value", "freight_ratio", "ship_year", "ship_month", "ship_date", "file_path")
                           ).withColumn("load_timestamp", current_timestamp())

order_items_enriched_df.limit(1).display()

In [0]:
order_items_enriched_df.createOrReplaceTempView('enriched_order_items_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.ORDER_ITEMS_ENRICHED`

In [0]:
%sql
MERGE INTO ORDERS_SALES.SILVER.ORDER_ITEMS_ENRICHED tgt
USING enriched_order_items_temp_vw src
ON (src.order_id = tgt.order_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.order_item_id = src.order_item_id,
    tgt.product_id = src.product_id,
    tgt.seller_id = src.seller_id,
    tgt.shipping_limit_date = src.shipping_limit_date
WHEN NOT MATCHED THEN
INSERT (order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value, is_price_non_positive,
is_freight_negative, is_ship_date_outlier, total_item_value, freight_ratio, ship_year, ship_month, ship_date, origin_file_path,
load_timestamp)
VALUES (src.order_id, src.order_item_id, src.product_id, src.seller_id, src.shipping_limit_date, src.price, src.freight_value,
src.is_price_non_positive, src.is_freight_negative, src.is_ship_date_outlier, src.total_item_value, src.freight_ratio, 
src.ship_year, src.ship_month, src.ship_date, src.file_path, current_timestamp)

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.ORDER_ITEMS_ENRICHED')